# All reconstruction results, per dataset
For every dataset with results, one table puts side by side:
- **between frames** (`recon_eval.py`): one 5-bin window between consecutive camera frames, whole sequences;
- **fixed-duration windows** (`recon_eval_tms.py`): 5 bins of a fixed length (2, 5, 10, 20 ms, ...), with fixed-ms or EVREAL-style bins.

- **vs real** (`agreement_*`, the main result): each method's reconstruction compared with the reconstruction from the **real recorded events** of the same scene. Same network, same window: the closer, the more realistic the simulated events.
- **vs GT** (`gt_*`, shown only with `SHOW_GT = True`): each reconstruction compared with the **camera frame**. The `real` row is the reference level; simulators were given these frames, so beating `real` here does not mean more realistic events.
- MSE and LPIPS: lower is better; SSIM: higher is better.

In every column the **top 3 methods per network are highlighted** (darkest = best, rank in brackets); `real` is never ranked. Values are the mean over sequences, ± the std over sequences. The newest finished run of each script is used per dataset (override with `RUNS`).

The last section shows the best saved frames for `OUR_METHOD` and saves each comparison image in that run's report folder (`figures/`).

In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Image as IPImage

REPO = Path.cwd().resolve()
while not (REPO / "benchmarks" / "recon_eval.py").is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "benchmarks"))
import recon_eval
import recon_figures

REAL, EVTWEEN, HIGHER = recon_eval.REAL, set(recon_eval.EVTWEEN), recon_eval.HIGHER_IS_BETTER
ORDER = {m: i for i, m in enumerate((REAL, *recon_eval.METHODS))}
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)


def latest(base, marker):
    """Newest run folder under base that contains marker (i.e. a finished run)."""
    runs = sorted(p for p in Path(base).glob("*") if (p / marker).is_file())
    return runs[-1] if runs else None


def resolve(run, base, marker):
    if run is None:
        return None
    if run == "latest":
        found = latest(base, marker)
        if found is None:
            print(f"No finished run under {base}")
        return found
    path = Path(run) if Path(run).is_absolute() else REPO / run
    if not (path / marker).is_file():
        raise FileNotFoundError(f"{path} has no {marker} (not a finished run?)")
    return path


def show_figure(path):
    if path is None:
        print("   (no saved frames for this one)")
        return
    print("   saved:", path.relative_to(REPO) if path.is_relative_to(REPO) else path)
    display(IPImage(filename=str(path)))

# ---- settings -------------------------------------------------------------
SPLIT = "test"
DATASETS = None            # None = every dataset with results, or e.g. ["eds", "hqevfi"]
RUNS = {}                  # overrides, e.g. {"eds": {"recon": "reports/recon/eds/test/2026...", "tms": None}}
SHOW_GT = True             # False: only the comparison with real events (tables and images)
OUR_METHOD = "evtween_v8_student"
TOP = 3                    # highlighted ranks per column
TABLE_METRICS = ("lpips", "ssim", "mse")   # e.g. ("lpips",) for narrower tables
SHOW_STD = True
BEST_METRIC = "agreement_lpips"   # agreement_* = vs real; gt_* = vs GT (needs SHOW_GT)
BEST_NETWORK = None               # "e2vid", "firenet", or None for the mean of both
BEST_TOP = 1                      # frames shown per setting
BEST_SETTINGS = None              # None = every setting, or e.g. ["between frames", "10 ms bins, fixed"]

METRICS = ("lpips", "ssim", "mse")
ARROW = {m: "↑" if HIGHER[m] else "↓" for m in METRICS}


def runs_for(ds):
    own = RUNS.get(ds, {})

    def pick(key, kind, marker):
        if key in own:
            return resolve(own[key], REPO / "reports" / kind / ds / SPLIT, marker) if own[key] is not None else None
        return latest(REPO / "reports" / kind / ds / SPLIT, marker)
    return dict(recon=pick("recon", "recon", "summary.csv"), tms=pick("tms", "recon_tms", "summary_all.csv"))


found = sorted({p.parent.name for kind in ("recon", "recon_tms") for p in (REPO / "reports" / kind).glob(f"*/{SPLIT}")})
datasets = DATASETS or found
RESULTS = {ds: runs_for(ds) for ds in datasets}
for ds, runs in RESULTS.items():
    print(ds)
    for k, v in runs.items():
        print(f"   {k:6s} {v if v else '-'}")

In [ ]:
def setting_label(bin_ms, bin_time):
    return f"{bin_ms:g} ms bins, {'fixed' if bin_time == 'ms' else 'EVREAL-style'}"


def long_rows(summary, setting, key):
    rows = []
    for _, r in summary.iterrows():
        for kind in ("agreement", "gt"):
            for m in METRICS:
                c = f"{kind}_{m}"
                if f"{c}_mean" in summary and pd.notna(r[f"{c}_mean"]):
                    rows.append(dict(setting=setting, key=key, network=r["network"], method=r["method"], kind=kind,
                                     metric=m, mean=r[f"{c}_mean"], std=r.get(f"{c}_std", np.nan)))
    return rows


def load(runs):
    rows, notes = [], []
    if runs["recon"]:
        rows += long_rows(pd.read_csv(runs["recon"] / "summary.csv"), "between frames", (0, 0.0))
        notes.append("between frames: one 5-bin window between consecutive camera frames")
    if runs["tms"]:
        s = pd.read_csv(runs["tms"] / "summary_all.csv")
        bins = json.loads((runs["tms"] / "config.json").read_text()).get("bins", 5)
        for _, g in s.groupby("config"):
            bms, bt = float(g["bin_ms"].iloc[0]), g["bin_time"].iloc[0]
            rows += long_rows(g, setting_label(bms, bt), (1 if bt == "ms" else 2, bms))
        notes.append(f"N ms bins: back-to-back windows of {bins} x N ms; 'fixed' = fixed bin edges, "
                     "'EVREAL-style' = bins stretched between the window's first and last event")
    return pd.DataFrame(rows), notes


SHADES = {1: "#9fc3ef", 2: "#c7dbf6", 3: "#e3edfa"}   # one blue hue, darkest = best


def ranked(mean, higher_by_col, skip_rows=()):
    ranks = pd.DataFrame(np.nan, index=mean.index, columns=mean.columns)
    for net in mean.index.get_level_values(0).unique():
        block = mean.loc[net]
        keep = [m for m in block.index if m not in skip_rows]
        for col in mean.columns:
            for m, v in block.loc[keep, col].dropna().rank(ascending=not higher_by_col[col], method="min").items():
                ranks.loc[(net, m), col] = v
    return ranks


def show_table(mean, std, higher_by_col, title, skip_rows=()):
    ranks = ranked(mean, higher_by_col, skip_rows)
    text = mean.copy().astype(object)
    css = pd.DataFrame("", index=mean.index, columns=mean.columns)
    for idx in mean.index:
        for col in mean.columns:
            v = mean.loc[idx, col]
            if pd.isna(v):
                text.loc[idx, col] = ""
                continue
            s = f"{v:.3f}"
            if SHOW_STD and std is not None and pd.notna(std.loc[idx, col]):
                s += f" ± {std.loc[idx, col]:.3f}"
            rk = ranks.loc[idx, col]
            if pd.notna(rk) and rk <= TOP:
                s += f" ({int(rk)})"
                css.loc[idx, col] = (f"background-color: {SHADES.get(int(rk), SHADES[3])}; color: #0b0b0b;"
                                     + (" font-weight: 700;" if rk == 1 else ""))
            elif idx[1] in skip_rows:
                css.loc[idx, col] = "color: #6b6a66; font-style: italic;"
            text.loc[idx, col] = s
    styler = text.style.apply(lambda _: css, axis=None).set_caption(title)

    def index_css(v):
        return "font-weight: 700; color: #1f5fa8;" if v == OUR_METHOD else ("font-weight: 600;" if v in EVTWEEN else "")
    try:
        styler = styler.map_index(index_css, axis=0, level=1)
    except AttributeError:  # pandas < 2.1
        styler = styler.applymap_index(index_css, axis=0, level=1)
    styler = styler.set_table_styles([
        {"selector": "caption", "props": "caption-side: top; font-size: 1.05em; font-weight: 600; text-align: left;"},
        {"selector": "th.col_heading", "props": "text-align: center;"},
        {"selector": "td", "props": "padding: 2px 8px; white-space: nowrap;"}])
    display(styler)


def study_table(long, kind):
    d = long[long["kind"] == kind]
    if d.empty:
        return None
    settings = d.drop_duplicates("setting").sort_values("key")["setting"].tolist()
    cols = [(s, f"{m.upper()} {ARROW[m]}") for s in settings for m in TABLE_METRICS
            if not d[(d["setting"] == s) & (d["metric"] == m)].empty]
    d = d.assign(col=list(zip(d["setting"], d["metric"].map(lambda m: f"{m.upper()} {ARROW[m]}"))))
    mean = d.pivot_table(index=["network", "method"], columns="col", values="mean", aggfunc="first").reindex(columns=cols)
    std = d.pivot_table(index=["network", "method"], columns="col", values="std", aggfunc="first").reindex(columns=cols)
    mean.columns = std.columns = pd.MultiIndex.from_tuples(cols, names=["setting", "metric"])
    rows = sorted(mean.index, key=lambda i: (i[0], ORDER.get(i[1], 99)))
    return mean.loc[rows], std.loc[rows], {c: c[1].endswith("↑") for c in cols}

## Results per dataset

In [ ]:
LEGEND = ("Highlighted: top 3 per column within each network (darkest = 1st, bold = best, rank in brackets). "
          f"real is the reference and is not ranked. {OUR_METHOD} is in blue, other Evtween variants in bold.")
for ds, runs in RESULTS.items():
    display(HTML(f"<h2 style='margin-top:1.5em'>{ds}</h2>"))
    long, notes = load(runs)
    if long.empty:
        print("No results for this dataset")
        continue
    for n in notes:
        print("•", n)
    print(LEGEND)
    t = study_table(long, "agreement")
    if t:
        show_table(*t, title=f"{ds}: vs real (agreement with the reconstruction from real events) - main result")
    if SHOW_GT:
        t = study_table(long, "gt")
        if t:
            show_table(*t, title=f"{ds}: vs GT (camera frame; real = reference level)", skip_rows=(REAL,))

## Best reconstructed frames for our method
For each dataset and setting: the saved frames (between frames) or windows (fixed duration) where `OUR_METHOD` scored best on `BEST_METRIC`, its rank among all methods there, and the comparison image (rows = networks; columns = real, every method, and GT when `SHOW_GT`). Each image is saved in that run's report folder (`figures/` or `<setting>/figures/`). Only saved frames can be shown, so this is the best among those.

In [ ]:
def best_saved(scores, key):
    metric = BEST_METRIC.split("_", 1)[1]
    s = scores[scores[BEST_METRIC].notna()]
    if BEST_NETWORK:
        s = s[s["network"] == BEST_NETWORK]
    table = s.groupby(["sequence", key, "method"])[BEST_METRIC].mean().unstack("method")
    if OUR_METHOD not in table:
        return None
    others = table.drop(columns=[REAL], errors="ignore")
    table["rank among methods"] = others.rank(axis=1, ascending=not HIGHER[metric], method="min")[OUR_METHOD]
    table["methods"] = others.notna().sum(axis=1)
    return table.sort_values(OUR_METHOD, ascending=not HIGHER[metric])[[OUR_METHOD, "rank among methods", "methods"]]


if BEST_METRIC.startswith("gt_") and not SHOW_GT:
    print("BEST_METRIC is a GT metric but SHOW_GT = False; using agreement_lpips")
    BEST_METRIC = "agreement_lpips"
tag = f"best_{OUR_METHOD}_{BEST_METRIC}"
for ds, runs in RESULTS.items():
    display(HTML(f"<h3>{ds}: best saved frames for {OUR_METHOD} by {BEST_METRIC} "
                 f"({'higher' if HIGHER[BEST_METRIC.split('_', 1)[1]] else 'lower'} is better, "
                 f"{BEST_NETWORK or 'mean of both networks'})</h3>"))
    if runs["recon"] and (BEST_SETTINGS is None or "between frames" in BEST_SETTINGS):
        if (runs["recon"] / "selection.csv").is_file():
            sel = pd.read_csv(runs["recon"] / "selection.csv")
            pf = pd.read_csv(runs["recon"] / "per_frame.csv").merge(sel[["sequence", "frame"]], on=["sequence", "frame"])
            top = best_saved(pf, "frame")
            print("between frames")
            if top is None:
                print(f"   {OUR_METHOD} is not in this run")
            else:
                display(top.head(BEST_TOP).round(4))
                for i, ((seq, k), _) in enumerate(top.head(BEST_TOP).iterrows(), 1):
                    show_figure(recon_figures.frame_figure(runs["recon"], seq, k, f"{tag}_{i}_{seq}_{int(k):06d}",
                                                           show_gt=SHOW_GT, title=f"best for {OUR_METHOD} #{i}"))
        else:
            print("between frames: no saved frames; make them from the stored metrics with\n"
                  f"   python benchmarks/recon_eval.py {ds} --devices cuda:0 cuda:0 --run-dir {runs['recon']} --save hybrid")
    if runs["tms"] and (runs["tms"] / "selection.csv").is_file():
        sel_all = pd.read_csv(runs["tms"] / "selection.csv")
        s = pd.read_csv(runs["tms"] / "summary_all.csv").drop_duplicates("config")
        s = s.assign(_k=list(zip(s["bin_time"] != "ms", s["bin_ms"]))).sort_values("_k")
        for _, row in s.iterrows():
            label, cfg = setting_label(float(row["bin_ms"]), row["bin_time"]), row["config"]
            sel = sel_all[sel_all["config"] == cfg]
            if (BEST_SETTINGS is not None and label not in BEST_SETTINGS) or sel.empty:
                continue
            pw = pd.read_csv(runs["tms"] / cfg / "per_window.csv").merge(sel[["sequence", "window"]],
                                                                          on=["sequence", "window"])
            top = best_saved(pw, "window")
            print(label)
            if top is None:
                print(f"   {OUR_METHOD} is not in this run")
                continue
            display(top.head(BEST_TOP).round(4))
            for i, ((seq, w), _) in enumerate(top.head(BEST_TOP).iterrows(), 1):
                show_figure(recon_figures.window_figure(runs["tms"], cfg, seq, w, f"{tag}_{i}_{seq}_w{int(w):06d}",
                                                        show_gt=SHOW_GT, title=f"best for {OUR_METHOD} #{i}"))